In [ ]:
# RQ1 - Dynamic Adaptation vs fixed coordination mechanisms
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'RQ1').exists() and (ROOT / 'Paper Results').exists():
    ROOT = ROOT / 'Paper Results'
summary_path = ROOT / 'RQ1' / 'summary_by_mechanism.csv'

if not summary_path.exists():
    print(f'RQ1 summary not found: {summary_path}')
    print('Run: python "Paper Results/run_rq1_experiments.py"')
else:
    df = pd.read_csv(summary_path)
    order = ['CE', 'SD', 'FD', 'DA']
    df['coord_label'] = pd.Categorical(df['coord_label'], categories=order, ordered=True)
    df = df.sort_values('coord_label')
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    plots = [
        ('avg_latency_s_mean', 'Average task latency (s)'),
        ('p95_latency_s_mean', 'P95 task latency (s)'),
        ('delivered_mean', 'Delivered tasks'),
        ('switch_count_mean', 'Coordination switches'),
    ]
    colors = {'CE': '#4C78A8', 'SD': '#F2B701', 'FD': '#E45756', 'DA': '#54A24B'}
    for ax, (metric, title) in zip(axes.flat, plots):
        ax.bar(df['coord_label'].astype(str), df[metric], color=[colors.get(x, '#999999') for x in df['coord_label'].astype(str)])
        ax.set_title(title)
        ax.grid(axis='y', alpha=0.25)
    fig.suptitle('RQ1: DA compared with fixed mechanisms', fontsize=14)
    fig.tight_layout()
    plt.show()
    display(df)


In [ ]:
# RQ2 - DA reaction to evolving operating conditions
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'RQ2').exists() and (ROOT / 'Paper Results').exists():
    ROOT = ROOT / 'Paper Results'
rq2 = ROOT / 'RQ2'
timeseries_path = rq2 / 'adaptation_state_timeseries.csv'
switch_path = rq2 / 'switch_timeline.csv'

if not timeseries_path.exists():
    print(f'RQ2 data not found: {timeseries_path}')
    print('Run: python "Paper Results/run_rq2_experiments.py"')
else:
    states = pd.read_csv(timeseries_path)
    switches = pd.read_csv(switch_path) if switch_path.exists() else pd.DataFrame()
    for col in ['sim_time', 'performance_score', 'score_latency', 'score_pending', 'score_failure', 'score_message_loss']:
        if col in states:
            states[col] = pd.to_numeric(states[col], errors='coerce')
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    first_runs = states.sort_values(['profile', 'run_id']).groupby('profile')['run_id'].first()
    for profile, run_id in first_runs.items():
        data = states[states['run_id'] == run_id].sort_values('sim_time')
        axes[0].plot(data['sim_time'] / 86400, data['performance_score'], label=profile)
    axes[0].axhline(0.35, color='#777777', linestyle='--', linewidth=1)
    axes[0].axhline(0.65, color='#333333', linestyle='--', linewidth=1)
    axes[0].set_title('Monitored score')
    axes[0].set_xlabel('Simulation day')
    axes[0].set_ylabel('S(W)')
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    if switches.empty:
        axes[1].text(0.5, 0.5, 'No switches recorded', ha='center', va='center')
        axes[1].set_axis_off()
    else:
        pivot = switches.pivot_table(index='profile', columns='selected_coord_label', values='switch_idx', aggfunc='count', fill_value=0)
        pivot.plot(kind='bar', stacked=True, ax=axes[1])
        axes[1].set_title('Switches by mechanism')
        axes[1].tick_params(axis='x', rotation=25)
        axes[1].grid(axis='y', alpha=0.25)
    state_frac = states.pivot_table(index='profile', columns='performance_state', values='sim_time', aggfunc='count', fill_value=0)
    state_frac = state_frac.div(state_frac.sum(axis=1), axis=0)
    state_frac.plot(kind='bar', stacked=True, ax=axes[2])
    axes[2].set_title('State distribution')
    axes[2].tick_params(axis='x', rotation=25)
    axes[2].grid(axis='y', alpha=0.25)
    fig.suptitle('RQ2: DA reaction to dynamic conditions', fontsize=14)
    fig.tight_layout()
    plt.show()
    display(pd.read_csv(rq2 / 'summary_by_profile.csv') if (rq2 / 'summary_by_profile.csv').exists() else states.head())


In [ ]:
# RQ3 - Runtime overhead of monitoring and adaptation
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'RQ3').exists() and (ROOT / 'Paper Results').exists():
    ROOT = ROOT / 'Paper Results'
rq3 = ROOT / 'RQ3'
summary_path = rq3 / 'overhead_summary.csv'
wall_path = rq3 / 'wall_time_summary.csv'
samples_path = rq3 / 'overhead_samples.csv'

if not summary_path.exists():
    print(f'RQ3 data not found: {summary_path}')
    print('Run: python "Paper Results/run_rq3_experiments.py"')
else:
    summary = pd.read_csv(summary_path)
    wall = pd.read_csv(wall_path) if wall_path.exists() else pd.DataFrame()
    samples = pd.read_csv(samples_path) if samples_path.exists() else pd.DataFrame()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    da = summary[summary['coord_label'] == 'DA']
    if not da.empty and 'adapt_overhead_ms_mean_mean' in da:
        da.pivot_table(index='n_robots', columns='item_spawning_time', values='adapt_overhead_ms_mean_mean').plot(marker='o', ax=axes[0])
        axes[0].set_title('Mean overhead per check')
        axes[0].set_ylabel('ms')
        axes[0].grid(alpha=0.25)
    if not samples.empty:
        samples[['total_overhead_ms', 'rtlola_query_ms', 'scoring_ms']].apply(pd.to_numeric, errors='coerce').mean().plot(kind='bar', ax=axes[1], color=['#4C78A8', '#F2B701', '#54A24B'])
        axes[1].set_title('Overhead breakdown')
        axes[1].set_ylabel('ms')
        axes[1].grid(axis='y', alpha=0.25)
    if not wall.empty and 'wall_time_s_mean' in wall:
        wall.plot(x='coord_label', y='wall_time_s_mean', kind='bar', ax=axes[2], legend=False, color='#72B7B2')
        axes[2].set_title('Wall-clock time')
        axes[2].set_ylabel('s')
        axes[2].grid(axis='y', alpha=0.25)
    fig.suptitle('RQ3: Runtime overhead', fontsize=14)
    fig.tight_layout()
    plt.show()
    display(summary)


In [ ]:
# RQ4 - Practical scalability limits
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'RQ4').exists() and (ROOT / 'Paper Results').exists():
    ROOT = ROOT / 'Paper Results'
summary_path = ROOT / 'RQ4' / 'scalability_summary.csv'

if not summary_path.exists():
    print(f'RQ4 data not found: {summary_path}')
    print('Run: python "Paper Results/run_rq4_experiments.py"')
else:
    df = pd.read_csv(summary_path)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for coord, data in df.groupby('coord_label'):
        wall = data.groupby('sim_days')['wall_time_s_mean'].mean().reset_index()
        axes[0, 0].plot(wall['sim_days'], wall['wall_time_s_mean'], marker='o', label=coord)
        rss = data.groupby('n_robots')['max_rss_mb_mean'].mean().reset_index()
        axes[0, 1].plot(rss['n_robots'], rss['max_rss_mb_mean'], marker='o', label=coord)
        trace = data.groupby('item_spawning_time')['total_output_size_mb_mean'].mean().reset_index().sort_values('item_spawning_time')
        axes[1, 0].plot(trace['item_spawning_time'], trace['total_output_size_mb_mean'], marker='o', label=coord)
        thr = data.groupby('n_robots')['events_per_wall_s_mean'].mean().reset_index()
        axes[1, 1].plot(thr['n_robots'], thr['events_per_wall_s_mean'], marker='o', label=coord)
    axes[0, 0].set_title('Wall-clock vs duration')
    axes[0, 1].set_title('Peak RSS vs agents')
    axes[1, 0].set_title('Trace size vs workload')
    axes[1, 0].invert_xaxis()
    axes[1, 1].set_title('Events per wall-clock second')
    for ax in axes.flat:
        ax.grid(alpha=0.25)
        ax.legend()
    fig.suptitle('RQ4: Scalability', fontsize=14)
    fig.tight_layout()
    plt.show()
    display(df)
